In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split


# ============================================================
# SETTINGS
# ============================================================

INPUT_FILE = "Kohomono_sil_wordlist_all.csv"   # Dataset
OUTDIR = "kohomono_final_figures"
DPI = 300

PHONEMIC_COL = "phonemic_form"          # column name 
TONE_COL = "tone_sequence"              # column name 

os.makedirs(OUTDIR, exist_ok=True)


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(INPUT_FILE)
df = df.dropna(subset=[PHONEMIC_COL]).copy()
df[PHONEMIC_COL] = df[PHONEMIC_COL].astype(str).str.strip()

if TONE_COL not in df.columns:
    df[TONE_COL] = ""


# ============================================================
# SYMBOL DEFINITIONS
# ============================================================

VOWELS = ["a", "e", "i", "o", "u", "ɔ", "á", "à", "é", "è", "í", "ì", "ó", "ò", "ú", "ù", "ɔ́", "ɔ̀"]

VOWEL_BASE = {
    "á": "a", "à": "a",
    "é": "e", "è": "e",
    "í": "i", "ì": "i",
    "ó": "o", "ò": "o",
    "ú": "u", "ù": "u",
    "ɔ́": "ɔ", "ɔ̀": "ɔ"
}

CONSONANTS = [
    "kp", "gb", "gw", "kw",
    "b", "d", "g", "k", "p", "t",
    "m", "n", "ŋ",
    "f", "s", "z", "h",
    "r", "l", "w", "y", "j",
    "β", "ɲ", "ʃ", "ʒ", "c", "ɟ"
]

SEGMENTS = sorted(CONSONANTS + VOWELS, key=len, reverse=True)


# ============================================================
# FUNCTIONS
# ============================================================

def normalize_vowel(seg):
    return VOWEL_BASE.get(seg, seg)


def tokenize_phonemic_form(form):
    form = str(form).strip()

    tokens = []
    i = 0

    while i < len(form):
        matched = False

        for seg in SEGMENTS:
            if form[i:i + len(seg)] == seg:
                tokens.append(normalize_vowel(seg))
                i += len(seg)
                matched = True
                break

        if not matched:
            ch = form[i]
            if ch not in [" ", ".", "-", "_", "/", "\\", "[", "]", "(", ")"]:
                tokens.append(normalize_vowel(ch))
            i += 1

    return tokens


def cv_skeleton(tokens):
    return "".join(["V" if t in ["a", "e", "i", "o", "u", "ɔ"] else "C" for t in tokens])


def classify_syllable(tokens):
    cv = cv_skeleton(tokens)

    if len(cv) == 0:
        return "mixed_other"

    if cv.startswith("V"):
        return "vowel_initial"

    if cv.startswith("CC"):
        return "consonant_cluster"

    if cv.endswith("C"):
        return "closed_final"

    if cv.startswith("C") and cv.endswith("V"):
        return "open_CV"

    return "mixed_other"


def extract_tone_sequence(form):
    form = str(form)

    tones = []

    for ch in form:
        if ch in ["́", "ˊ", "á", "é", "í", "ó", "ú"]:
            tones.append("H")
        elif ch in ["̀", "ˋ", "à", "è", "ì", "ò", "ù"]:
            tones.append("L")

    return "".join(tones) if tones else "none"


def add_panel_label(ax, label):
    ax.text(
        -0.08,
        1.08,
        label,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        va="top",
        ha="left"
    )


# ============================================================
# PROCESS DATA
# ============================================================

df["tokens"] = df[PHONEMIC_COL].apply(tokenize_phonemic_form)
df["word_length_segments"] = df["tokens"].apply(len)
df["cv_skeleton"] = df["tokens"].apply(cv_skeleton)
df["syllable_class"] = df["tokens"].apply(classify_syllable)

if df[TONE_COL].astype(str).str.strip().eq("").all():
    df["tone_sequence"] = df[PHONEMIC_COL].apply(extract_tone_sequence)
else:
    df["tone_sequence"] = df[TONE_COL].astype(str).str.strip().replace({"nan": "none", "": "none"})


all_segments = [seg for toks in df["tokens"] for seg in toks]

seg_freq = pd.DataFrame(Counter(all_segments).items(), columns=["segment", "count"])
seg_freq = seg_freq.sort_values("count", ascending=False)

vowel_freq = seg_freq[seg_freq["segment"].isin(["a", "e", "i", "o", "u", "ɔ"])].copy()
consonant_freq = seg_freq[~seg_freq["segment"].isin(["a", "e", "i", "o", "u", "ɔ"])].copy()

cv_freq = df["cv_skeleton"].value_counts().reset_index()
cv_freq.columns = ["cv_skeleton", "count"]

tone_freq = df["tone_sequence"].value_counts().reset_index()
tone_freq.columns = ["tone_sequence", "count"]

class_freq = df["syllable_class"].value_counts().reset_index()
class_freq.columns = ["syllable_class", "count"]


# ============================================================
# FEATURES
# ============================================================

X = pd.DataFrame(index=df.index)

X["word_length_segments"] = df["word_length_segments"]
X["syllable_count_approx"] = df["cv_skeleton"].str.count("V")
X["tone_count"] = df["tone_sequence"].apply(lambda x: 0 if x == "none" else len(str(x)))
X["starts_with_vowel"] = df["cv_skeleton"].str.startswith("V").astype(int)
X["ends_with_consonant"] = df["cv_skeleton"].str.endswith("C").astype(int)
X["has_initial_consonant_cluster"] = df["cv_skeleton"].str.startswith("CC").astype(int)

top_segments = seg_freq.head(15)["segment"].tolist()

for seg in top_segments:
    X[f"seg_{seg}"] = df["tokens"].apply(lambda toks: int(seg in toks))

X = X.fillna(0)
y = df["syllable_class"]


# ============================================================
# PCA AND CLUSTERING
# ============================================================

pca = PCA(n_components=2)
pca_values = pca.fit_transform(X)

pca_df = pd.DataFrame(pca_values, columns=["PC1", "PC2"])
pca_df["syllable_class"] = y.values

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
pca_df["cluster"] = kmeans.fit_predict(pca_df[["PC1", "PC2"]])

sample_df = X.sample(n=min(30, len(X)), random_state=42)
Z = linkage(sample_df, method="ward")
labels = sample_df.index.astype(str).tolist()


# ============================================================
# RANDOM FOREST
# ============================================================

print("Syllable class counts before split:")
print(y.value_counts())

rare_classes = y.value_counts()[y.value_counts() < 2].index

if len(rare_classes) > 0:
    print("Rare classes merged into mixed_other:", list(rare_classes))
    y = y.replace(rare_classes, "mixed_other")
    pca_df["syllable_class"] = y.values

print("Syllable class counts used for model:")
print(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

importances = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)


# ============================================================
# FIGURE 1: VOWELS AND CONSONANTS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].bar(vowel_freq["segment"], vowel_freq["count"])
axes[0].set_title("Observed vowel frequencies")
axes[0].set_xlabel("Vowel")
axes[0].set_ylabel("Count")
add_panel_label(axes[0], "A")

top_cons = consonant_freq.head(20)
axes[1].bar(top_cons["segment"], top_cons["count"])
axes[1].set_title("Top observed consonant frequencies")
axes[1].set_xlabel("Consonant")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)
add_panel_label(axes[1], "B")

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Figure_1_vowel_and_consonant_frequencies.png"), dpi=DPI)
plt.close()


# ============================================================
# FIGURE 2: CV SKELETON AND WORD LENGTH
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top_cv = cv_freq.head(20)
axes[0].bar(top_cv["cv_skeleton"], top_cv["count"])
axes[0].set_title("Top C/V skeleton frequencies")
axes[0].set_xlabel("C/V skeleton")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=60)
add_panel_label(axes[0], "A")

axes[1].hist(
    df["word_length_segments"],
    bins=range(1, int(df["word_length_segments"].max()) + 2),
    edgecolor="black"
)
axes[1].set_title("Word length distribution based on segment count")
axes[1].set_xlabel("Word length (segments)")
axes[1].set_ylabel("Number of words")
add_panel_label(axes[1], "B")

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Figure_2_cv_skeleton_and_word_length.png"), dpi=DPI)
plt.close()


# ============================================================
# FIGURE 3: TONE AND TBU
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top_tone = tone_freq.head(20)
axes[0].bar(top_tone["tone_sequence"], top_tone["count"])
axes[0].set_title("Top tone sequence frequencies")
axes[0].set_xlabel("Tone sequence")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=60)
add_panel_label(axes[0], "A")

tbu_counts = df["tone_sequence"].apply(lambda x: 0 if x == "none" else len(str(x))).value_counts().sort_index()
axes[1].bar(tbu_counts.index.astype(str), tbu_counts.values)
axes[1].set_title("Tone-bearing units per word")
axes[1].set_xlabel("TBUs per word")
axes[1].set_ylabel("Word count")
add_panel_label(axes[1], "B")

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Figure_3_tone_sequences_and_tbus.png"), dpi=DPI)
plt.close()


# ============================================================
# FIGURE 4: SYLLABLE CLASS AND PCA
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].bar(class_freq["syllable_class"], class_freq["count"])
axes[0].set_title("Broad syllable class distribution")
axes[0].set_xlabel("Syllable class")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)
add_panel_label(axes[0], "A")

for cls in sorted(pca_df["syllable_class"].dropna().unique()):
    sub = pca_df[pca_df["syllable_class"] == cls]
    axes[1].scatter(sub["PC1"], sub["PC2"], label=cls, alpha=0.65, s=25)

axes[1].set_title(
    f"PCA of Kohomono words\n"
    f"PC1={pca.explained_variance_ratio_[0] * 100:.1f}%, "
    f"PC2={pca.explained_variance_ratio_[1] * 100:.1f}%"
)
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].legend(fontsize=8)
add_panel_label(axes[1], "B")

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Figure_4_syllable_class_and_pca.png"), dpi=DPI)
plt.close()


# ============================================================
# FIGURE 5: K-MEANS AND DENDROGRAM
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for cl in sorted(pca_df["cluster"].unique()):
    sub = pca_df[pca_df["cluster"] == cl]
    axes[0].scatter(sub["PC1"], sub["PC2"], label=f"Cluster {cl}", alpha=0.70, s=25)

axes[0].set_title("K-means clusters in PCA space")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend(fontsize=8)
add_panel_label(axes[0], "A")

dendrogram(
    Z,
    labels=labels,
    leaf_rotation=90,
    leaf_font_size=7,
    ax=axes[1]
)

axes[1].set_title("Hierarchical clustering dendrogram")
axes[1].set_xlabel("Sampled word")
axes[1].set_ylabel("Distance")
add_panel_label(axes[1], "B")

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Figure_5_kmeans_and_dendrogram.png"), dpi=DPI)
plt.close()


# ============================================================
# FIGURE 6: RANDOM FOREST
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ---- Figure 6A: Feature importance ----
top_imp = importances.head(20)

axes[0].bar(top_imp["feature"], top_imp["importance"])
axes[0].set_title("Random Forest feature importance")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("Importance")
axes[0].tick_params(axis="x", rotation=60, labelsize=8)
add_panel_label(axes[0], "A")


# ---- Figure 6B: Confusion matrix ----
label_map = {
    "vowel_initial": "V-initial",
    "open_CV": "Open-CV",
    "closed_final": "Closed-final",
    "consonant_cluster": "CC-cluster",
    "mixed_other": "Mixed"
}

y_test_short = pd.Series(y_test).replace(label_map)
y_pred_short = pd.Series(y_pred).replace(label_map)

ConfusionMatrixDisplay.from_predictions(
    y_test_short,
    y_pred_short,
    ax=axes[1],
    cmap="viridis",
    colorbar=True,
    xticks_rotation=25
)

axes[1].set_title(
    f"Random Forest confusion matrix\nAccuracy = {accuracy:.3f}",
    fontsize=11
)

axes[1].set_xlabel("Predicted class", fontsize=10)
axes[1].set_ylabel("True class", fontsize=10)
axes[1].tick_params(axis="x", labelsize=8)
axes[1].tick_params(axis="y", labelsize=8)

add_panel_label(axes[1], "B")

plt.tight_layout()
plt.savefig(
    os.path.join(OUTDIR, "Figure_6_feature_importance_and_confusion_matrix.png"),
    dpi=DPI,
    bbox_inches="tight"
)
plt.close()


# ============================================================
# SAVE CSV OUTPUTS
# ============================================================

seg_freq.to_csv(os.path.join(OUTDIR, "segment_frequencies.csv"), index=False)
vowel_freq.to_csv(os.path.join(OUTDIR, "vowel_frequencies.csv"), index=False)
consonant_freq.to_csv(os.path.join(OUTDIR, "consonant_frequencies.csv"), index=False)
cv_freq.to_csv(os.path.join(OUTDIR, "cv_skeleton_frequencies.csv"), index=False)
tone_freq.to_csv(os.path.join(OUTDIR, "tone_sequence_frequencies.csv"), index=False)
class_freq.to_csv(os.path.join(OUTDIR, "syllable_class_distribution.csv"), index=False)
importances.to_csv(os.path.join(OUTDIR, "random_forest_feature_importance.csv"), index=False)

print("Done.")
print(f"Figures saved in: {OUTDIR}")
print(f"Random Forest accuracy: {accuracy:.4f}")

Syllable class counts before split:
syllable_class
vowel_initial        518
open_CV              278
closed_final         123
consonant_cluster     36
Name: count, dtype: int64
Syllable class counts used for model:
syllable_class
vowel_initial        518
open_CV              278
closed_final         123
consonant_cluster     36
Name: count, dtype: int64
Done.
Figures saved in: kohomono_final_figures
Random Forest accuracy: 1.0000
